# Validación · Problema 3 — Combinar cifras y operaciones

---
## Pregunta 1 — ¿Cuántas posibilidades hay sin / con restricciones?

`celdas 2–4` · Enunciado ✔ · Texto ✔ · Código ✔ · 1ª parte **obligatoria (\*)** — 2ª parte opcional

### Validación

La aritmética es correcta en ambos casos: sin restricciones (repitiendo cifras y operadores) 9⁵×4⁴ =
**15.116.544**; con restricciones (variaciones de 5 cifras entre 9, sin repetir, combinadas con las 4!
ordenaciones de los operadores) V(9,5)×4! = 15.120×24 = **362.880**, verificado por código con `math.perm`
y `math.factorial`. Es una respuesta sólida; solo le falta citar el ejemplo del enunciado como
justificación explícita de por qué se usan 5 cifras y no 9.

**Opción 1 — fórmula combinatoria cerrada (así está implementada en Resuelto.ipynb)**

Expresar el conteo con las fórmulas de variaciones y permutaciones, y verificarlo con las funciones de `math`:

In [ ]:
import math

CIFRAS = tuple(range(1, 10))
OPERACIONES = ('+', '-', '*', '/')
K = 5

sin_restricciones = len(CIFRAS) ** K * len(OPERACIONES) ** (K - 1)
con_restricciones = math.perm(len(CIFRAS), K) * math.factorial(len(OPERACIONES))

print(f"Sin restricciones: 9^5 * 4^4 = {sin_restricciones:,}")
print(f"Con restricciones: P(9,5) * 4! = {con_restricciones:,}")

**Justificación.** Esta es la opción más recomendable porque expresa el resultado en función de fórmulas combinatorias con
nombre propio — variaciones sin repetición V(n,k) y permutaciones m! — en vez de contarlo "a ciegas". Eso
demuestra que el razonamiento matemático se entendió, no solo que se llegó al número correcto por prueba y
error. Se ejecuta en tiempo O(1): no importa si n o k crecieran, el cálculo sigue siendo instantáneo, lo que
la hace directamente reutilizable en la Pregunta 5 (complejidad) sin cambiar nada. Su único requisito es
conocer de antemano qué fórmula combinatoria corresponde a "elegir y ordenar k de n elementos", que es
precisamente el conocimiento que esta pregunta del rubric busca comprobar.

**Opción 2 — enumeración exhaustiva y conteo real**

En vez de aplicar la fórmula, generar de verdad todas las variaciones posibles y contarlas:

In [ ]:
import itertools

def contar_por_enumeracion(cifras=CIFRAS, k=K, operaciones=OPERACIONES):
    total = 0
    for _ in itertools.permutations(cifras, k):
        for _ in itertools.permutations(operaciones):
            total += 1
    return total

print(f"Con restricciones (contado uno a uno): {contar_por_enumeracion():,}")

**Justificación.** Esta alternativa tiene la ventaja de no depender de recordar ninguna fórmula: si el razonamiento
combinatorio genera dudas, generar y contar literalmente todas las expresiones es una forma robusta de
llegar al número correcto y de verificar por otra vía que la fórmula cerrada no tiene ningún error de
planteamiento. Es especialmente útil como comprobación cruzada de la opción 1. Su límite principal es de
eficiencia: cuesta O(n⁵) tiempo y recorre 362.880 elementos solo para obtener un número que la fórmula
cerrada calcula en O(1), y si n creciera bastante (por ejemplo n=30) esta vía dejaría de ser práctica mucho
antes que la fórmula. Tampoco dice nada por sí sola sobre el porqué combinatorio del resultado.

**Opción 3 — cálculo manual paso a paso, sin funciones de `math`**

Reproducir la regla del producto con un bucle explícito, sin usar `math.perm` ni `math.factorial`, para dejar visible cada factor:

In [ ]:
def contar_paso_a_paso(n_cifras=9, k=K, n_ops=4):
    variaciones = 1
    for i in range(k):                 # 9 * 8 * 7 * 6 * 5
        variaciones *= (n_cifras - i)
    ordenes_operadores = 1
    for i in range(n_ops):             # 4 * 3 * 2 * 1
        ordenes_operadores *= (n_ops - i)
    return variaciones * ordenes_operadores

print(f"Con restricciones (paso a paso): {contar_paso_a_paso():,}")

**Justificación.** Esta variante hace explícito, factor a factor, de dónde sale cada número (9·8·7·6·5 para las cifras,
4·3·2·1 para los operadores) sin apoyarse en `math.perm`/`math.factorial` ni en generar las expresiones
reales. Es pedagógicamente transparente y útil si se quiere mostrar el desglose completo del cálculo en el
propio código, no solo en el texto. Su desventaja frente a las otras dos es que reimplementa a mano una
lógica que la librería estándar ya ofrece de forma probada y legible (`math.perm`, `math.factorial`), así
que es más código para mantener y algo más fácil de introducir un error de índice (por ejemplo, en el rango
del bucle) que al llamar directamente a las funciones combinatorias de la opción 1.

---
## Pregunta 2 — Estructura de datos: justificación

`celdas 5–7` · Enunciado ✔ · Texto ✔ · Código ✔ · **Obligatoria (\*)**

### Validación

Respuesta fuerte. Propone tuplas `(cifras, operaciones)` para cada expresión candidata, un diccionario
`valor → expresión` para acumular resultados en O(1), y documenta un diseño descartado (texto + `eval()`)
con motivos técnicos concretos: coste de reparseo y riesgo de imprecisión de coma flotante. Es exactamente
lo que pide el rubric ("es posible que hayas elegido una al principio y veas la necesidad de cambiar,
argumenta").

**Opción 1 — tuplas + diccionario + aritmética racional exacta a mano (así está implementada)**

Representar cada expresión como dos tuplas y evaluarla con una función que hace aritmética entera exacta (numerador/denominador), sin `eval` ni `Fraction`:

In [ ]:
def evaluar_exacto(cifras, operaciones):
    total_num, total_den = 0, 1
    signo = 1
    term_num, term_den = cifras[0], 1
    for op, c in zip(operaciones, cifras[1:]):
        if op == '*':
            term_num *= c
        elif op == '/':
            term_den *= c
        else:
            total_num = total_num * term_den + signo * term_num * total_den
            total_den *= term_den
            signo = 1 if op == '+' else -1
            term_num, term_den = c, 1
    total_num = total_num * term_den + signo * term_num * total_den
    total_den *= term_den
    return total_num, total_den

resultados = {}   # valor_entero -> (cifras, operaciones)

**Justificación.** Es la opción más recomendable para este problema: las tuplas son inmutables, ligeras y encajan
directamente con lo que devuelve `itertools.permutations`; el diccionario deja insertar y comprobar
pertenencia en O(1); y la aritmética racional a mano evita por completo tanto el coste de reparsear texto
en cada evaluación como el riesgo de que `6/3` no dé exactamente `2.0` por redondeo de coma flotante. Es la
única de las tres que resuelve simultáneamente los dos problemas reales del enunciado (rendimiento y
precisión) sin añadir ninguna dependencia externa. Su coste es de legibilidad: la función `evaluar_exacto`
es más difícil de leer a primera vista que una simple llamada a `eval()` o a `Fraction`.

**Opción 2 — `fractions.Fraction` por cifra**

Dejar que la clase `Fraction` de la librería estándar gestione la exactitud automáticamente:

In [ ]:
from fractions import Fraction
import operator

OPS = {'+': operator.add, '-': operator.sub, '*': operator.mul, '/': operator.truediv}

def evaluar_con_fraction(cifras, operaciones):
    # Nota: para respetar la precedencia (* / antes que + -) haria falta
    # agrupar primero los terminos; aqui se muestra la idea basica de
    # encadenar Fraction para operaciones sin precedencia mixta.
    valor = Fraction(cifras[0])
    for op, c in zip(operaciones, cifras[1:]):
        valor = OPS[op](valor, Fraction(c))
    return valor

**Justificación.** Esta alternativa es igual de correcta en cuanto a precisión: `Fraction` nunca introduce error de redondeo,
así que resuelve el mismo problema que la opción 1 sin tener que programar a mano la aritmética de
numerador/denominador. Es más legible y más rápida de escribir, y sería la primera opción razonable si el
proyecto no tuviera un requisito de rendimiento estricto. Su desventaja es de velocidad: `Fraction`
normaliza con el máximo común divisor en cada operación aritmética, lo que introduce un overhead medible
cuando se repite más de un millón de veces (362.880 expresiones × 4 operaciones), además de que aquí habría
que añadir lógica extra para respetar la precedencia de operadores, que la opción 1 ya resuelve directamente.

**Opción 3 — cadena de texto + `eval()`, tal como sugiere el propio enunciado**

Construir la expresión como texto y evaluarla directamente, con una comprobación de entero por tolerancia:

In [ ]:
def evaluar_con_eval(cifras, operaciones):
    partes = [str(cifras[0])]
    for op, c in zip(operaciones, cifras[1:]):
        partes += [op, str(c)]
    expr = ' '.join(partes)
    valor = eval(expr)
    es_entero = abs(valor - round(valor)) < 1e-9   # necesario: eval trabaja en coma flotante
    return valor, es_entero

**Justificación.** Es la opción más simple de escribir y la más directa a partir del propio hint del enunciado, lo que la hace
muy útil para un primer prototipo o para validar de forma independiente los resultados del método elegido
como definitivo. Es perfectamente correcta si se le añade la comprobación de tolerancia (`abs(valor -
round(valor)) < 1e-9`), que aquí es obligatoria y no opcional, precisamente por trabajar en coma flotante.
Sus dos límites frente a las opciones 1 y 2 son el coste de reparsear la cadena en cada una de las
362.880×24 evaluaciones y el riesgo de que, en algún caso límite con divisiones encadenadas, el margen de
tolerancia elegido no sea suficiente.

---
## Pregunta 3 — Función objetivo · ¿maximización o minimización?

`celdas 8–10` · Enunciado ✔ · Texto ✔ · Código ✔ · **Obligatoria (\*) ×2**

### Validación

Esta es la pregunta que más rigor exige: el problema en sí es pequeño y no es "de verdad" un problema de
optimización clásico, así que una respuesta plana de una línea se penaliza más aquí que en cualquier otra
pregunta. El notebook distingue correctamente **dos naturalezas distintas**: (1) un problema de
enumeración/decisión (¿qué enteros pertenecen a la imagen de f?) y (2) dos problemas de optimización
explícitos e independientes (maximizar y minimizar f). Le falta nombrar una tercera faceta — la cobertura
del rango, ya usada en la Pregunta 9 — para estar completa.

**Opción 1 — doble naturaleza: decisión/enumeración + dos optimizaciones (max y min), + cobertura**

Definir f con la precedencia estándar y evaluarla sobre una expresión concreta:

In [ ]:
def f(cifras, operaciones):
    num, den = evaluar_exacto(cifras, operaciones)
    return num / den

# f admite tres lecturas sobre el mismo espacio de 362.880 expresiones:
#  1) decision/enumeracion: que enteros pertenecen a la imagen de f (restringida a resultados enteros)
#  2) optimizacion:        maximizar f  y  minimizar f  (dos busquedas independientes)
#  3) cobertura:           de los enteros entre el minimo y el maximo, cuales se alcanzan realmente
print(f((4, 2, 6, 3, 1), ('+', '-', '/', '*')))

**Justificación.** Es la lectura más completa del problema porque explica *por qué* el rubric pregunta a la vez por la función
objetivo y por max/min: no hay una única respuesta simple posible, porque el enunciado en realidad plantea
tres preguntas relacionadas pero distintas sobre el mismo espacio de soluciones. Nombrar también la tercera
faceta (cobertura del rango) deja ver que se entendió por qué el algoritmo con poda de la Pregunta 6 no es
suficiente por sí solo para responder "qué enteros son alcanzables" — solo responde a las dos optimizaciones,
no a la cobertura. Es la opción que mejor prepara el terreno para las preguntas siguientes del notebook.

**Opción 2 — objetivo escalarizado: maximizar |f|**

Reformular como una única optimización sobre el valor absoluto, deduciendo el signo aparte:

In [ ]:
def f_escalarizada(cifras, operaciones):
    num, den = evaluar_exacto(cifras, operaciones)
    return abs(num / den)

# maximizar f_escalarizada da el extremo de mayor magnitud;
# el signo real (si es el maximo o el -minimo) se recupera aparte

**Justificación.** Esta forma tiene la ventaja de reducir el problema a una única búsqueda de optimización (maximizar una sola
función) en lugar de dos búsquedas independientes, lo que simplificaría el código de poda a una sola cota.
Es una reformulación matemáticamente válida y perfectamente defendible. Su desventaja frente a la opción 1
es que oscurece el resultado que el enunciado realmente pide reportar de forma directa — el máximo y el
mínimo como dos cifras concretas, con su signo — y no es la forma en que luego se implementa el propio
backtracking del notebook, que busca max y min por separado; usarla obligaría a adaptar el resto del código
para mantener la coherencia.

**Opción 3 — f definida con foco en maximización, sin desarrollar la doble naturaleza**

Definir f y quedarse con una única clasificación explícita:

In [ ]:
def f(cifras, operaciones):
    num, den = evaluar_exacto(cifras, operaciones)
    return num / den

# Es un problema de maximizacion: se busca la combinacion de cifras y
# operaciones que produce el mayor valor entero posible de f.

**Justificación.** Esta opción define correctamente la función objetivo y acierta con una de las dos búsquedas que pide el
problema (maximización), lo cual no es incorrecto. Tiene sentido como primer paso o como respuesta rápida
cuando el tiempo apremia. Su límite frente a las opciones 1 y 2 es que dejar fuera la minimización (que el
propio rubric pide igual de explícitamente) y la naturaleza de enumeración del problema deja la respuesta
solo a mitad de camino: responde a una parte de la pregunta con precisión, pero no aprovecha la oportunidad
de mostrar que se entendió por qué el enunciado pide clasificar el problema y no solo definir f().

---
## Pregunta 4 — Diseño del algoritmo de fuerza bruta

`celdas 11–13` · Enunciado ✔ · Texto ✔ · Código ✔ · Opcional

### Validación

Correcto y exhaustivo — recorre las 362.880 expresiones con `itertools.permutations` y evalúa cada una con
aritmética racional exacta (no `eval`). El pseudocódigo previo a la implementación es claro y coincide con
el código real. Falta una frase explícita de cierre aclarando que las optimizaciones de implementación
(aritmética exacta, generar los 24 órdenes una sola vez) bajan la constante, no la clase de complejidad —
para no confundir esta versión con la mejorada de la Pregunta 6.

**Opción 1 — doble bucle sobre `itertools.permutations` (así está implementada)**

Recorrer todas las variaciones de cifras y todos los órdenes de operadores con dos bucles anidados:

In [ ]:
import itertools

def fuerza_bruta(cifras_disponibles=CIFRAS, k=K):
    resultados = {}
    ops_perms = list(itertools.permutations(OPERACIONES))
    for cifras in itertools.permutations(cifras_disponibles, k):
        for ops in ops_perms:
            num, den = evaluar_exacto(cifras, ops)
            if num % den == 0:
                valor = num // den
                if valor not in resultados:
                    resultados[valor] = (cifras, ops)
    return resultados

**Justificación.** Es la opción más recomendable como fuerza bruta de referencia: `itertools.permutations` está implementado en
C, por lo que iterar sobre él es mucho más rápido que cualquier equivalente escrito a mano en Python puro;
además es un generador, así que no materializa las 362.880 combinaciones en memoria a la vez, y los 24
órdenes de operadores se calculan una sola vez fuera del bucle principal en lugar de recalcularse en cada
iteración. Es exhaustiva (no descarta ninguna rama) y sirve como oráculo de referencia frente al algoritmo
con poda. Su límite es conceptual, no de implementación: sigue siendo O(n⁵), y no debe confundirse con una
mejora algorítmica real.

**Opción 2 — cadena de texto + `eval()`, literal al enunciado**

Construir cada expresión como texto y evaluarla con `eval()`, tal y como sugiere el propio enunciado:

In [ ]:
import itertools

def fuerza_bruta_eval(cifras_disponibles=CIFRAS, k=K):
    resultados = {}
    for cifras in itertools.permutations(cifras_disponibles, k):
        for ops in itertools.permutations(OPERACIONES):
            partes = [str(cifras[0])]
            for op, c in zip(ops, cifras[1:]):
                partes += [op, str(c)]
            valor = eval(' '.join(partes))
            if abs(valor - round(valor)) < 1e-9:
                v = round(valor)
                if v not in resultados:
                    resultados[v] = (cifras, ops)
    return resultados

**Justificación.** Esta versión es la más directa a partir del propio enunciado, que sugiere explícitamente usar `eval()`, y
por eso es la más fácil de explicar a alguien que no conozca aritmética racional con fracciones. Es
perfectamente correcta con la comprobación de tolerancia añadida. Su desventaja frente a la opción 1 es de
rendimiento: reparsear una cadena de texto en cada una de las 362.880×24 evaluaciones es más lento que
operar directamente con enteros, y arrastra el riesgo — ya discutido en la Pregunta 2 — de que la
comparación por tolerancia falle en algún caso límite con divisiones encadenadas.

**Opción 3 — backtracking recursivo explícito, sin poda**

Construir la expresión posición a posición con una función recursiva, en vez de dos bucles sobre `itertools`:

In [ ]:
def fuerza_bruta_recursiva(pool=CIFRAS, k=K):
    resultados = {}
    n = len(pool)

    def rec(usados, cifras_seq, ops_libres, ops_seq):
        if len(cifras_seq) == k:
            num, den = evaluar_exacto(tuple(cifras_seq), tuple(ops_seq))
            if num % den == 0:
                resultados.setdefault(num // den, (tuple(cifras_seq), tuple(ops_seq)))
            return
        for i in range(n):
            if i in usados:
                continue
            usados.add(i); cifras_seq.append(pool[i])
            for op in ops_libres:
                ops_seq.append(op)
                rec(usados, cifras_seq, ops_libres - {op}, ops_seq)
                ops_seq.pop()
            cifras_seq.pop(); usados.remove(i)

    for i in range(n):
        rec({i}, [pool[i]], set(OPERACIONES), [])
    return resultados

**Justificación.** Esta variante recorre exactamente el mismo espacio de 362.880 expresiones de forma igual de exhaustiva y
correcta, pero construyéndolo cifra a cifra en vez de generar permutaciones completas de antemano. Su
verdadera ventaja no es de rendimiento sino de diseño: es la misma estructura recursiva sobre la que se
construye directamente el algoritmo con poda de la Pregunta 6 (basta con añadir la comprobación de la cota
en el punto adecuado), así que sirve de puente pedagógico natural entre ambas preguntas. Su desventaja es de
velocidad: en Python, cada llamada a función recursiva tiene más overhead que la iteración de
`itertools.permutations`, implementada en C, así que para fuerza bruta pura (sin poda) es más lenta que la
opción 1 sin aportar ninguna ventaja adicional todavía.

---
## Pregunta 5 — Complejidad del algoritmo de fuerza bruta

`celdas 14–16` · Enunciado ✔ · Texto ✔ · Código + gráfica ✔ · Opcional

### Validación

Derivación correcta y cuidada: identifica que k=5 y m=4 son constantes del problema — no crecen con n —,
así que T(n) = V(n,5)×4! = O(n⁵), polinómica, no factorial. La afirmación se contrasta empíricamente
ejecutando el algoritmo para n=9,11,13,15 y superponiendo una curva de referencia n⁵ en una gráfica. Buena
práctica adicional: aclara que en el problema original (n=9 fijo) la ejecución es técnicamente O(1), y que
generalizar a n variable es una elección deliberada para poder hablar de complejidad.

**Opción 1 — derivación analítica O(n⁵) + validación empírica con gráfica (así está)**

Medir el tiempo real de la fuerza bruta para varios n y compararlo con una curva de referencia n⁵:

In [ ]:
import time
import matplotlib.pyplot as plt

def medir_fuerza_bruta(n_valores):
    tiempos = []
    for n in n_valores:
        pool = tuple(range(1, n + 1))
        t0 = time.perf_counter()
        fuerza_bruta(pool)
        tiempos.append(time.perf_counter() - t0)
    return tiempos

n_valores = [9, 11, 13, 15]
tiempos_bf = medir_fuerza_bruta(n_valores)
referencia_n5 = [tiempos_bf[0] * (n / n_valores[0]) ** 5 for n in n_valores]

plt.plot(n_valores, tiempos_bf, 'o-', label='tiempo medido')
plt.plot(n_valores, referencia_n5, '--', label='referencia O(n^5)')
plt.xlabel('n'); plt.ylabel('tiempo (s)'); plt.legend(); plt.show()

**Justificación.** Es la opción más sólida porque combina teoría con evidencia: la derivación V(n,5)×4! = O(n⁵) es la única
forma de *probar* la complejidad, y la gráfica frente a una curva de referencia n⁵ es la única forma de
detectar si esa teoría se equivocó en algo (por ejemplo, una constante oculta enorme que la haga
impracticable mucho antes de lo previsto). Reconocer que k=5 y m=4 son constantes del problema, y no
parámetros que crecen con n, es el matiz que distingue un análisis de complejidad correcto de uno
apresurado. Su único costo es tener que ejecutar realmente el algoritmo varias veces para generar los datos
de la gráfica, lo cual toma unos segundos.

**Opción 2 — ajuste puramente empírico del exponente de crecimiento**

Medir tiempos para varios n y estimar el exponente ajustando una recta en escala log-log, sin derivar la fórmula a mano:

In [ ]:
import numpy as np

ns = [9, 11, 13, 15]
tiempos = medir_fuerza_bruta(ns)
pendiente = np.polyfit(np.log(ns), np.log(tiempos), 1)[0]
print(f"Exponente estimado empíricamente: {pendiente:.2f}  (esperado: ~5)")

**Justificación.** Esta alternativa es útil cuando la fórmula combinatoria exacta no es evidente a simple vista: en vez de
derivarla, se infiere el exponente de crecimiento a partir de mediciones reales, ajustando una recta en
escala logarítmica. Es una técnica legítima y de uso común en ingeniería del rendimiento. Su límite frente a
la opción 1 es que el exponente estimado es ruidoso — depende del calentamiento de caché, la resolución del
reloj del sistema y el recolector de basura de Python — y nunca constituye una demostración matemática, solo
un indicio que respalda (o contradice) una hipótesis previa; por eso funciona mejor como complemento de la
derivación analítica que como sustituto.

**Opción 3 — conteo exacto de operaciones para el caso fijo (n=9), sin generalizar**

Calcular el número exacto de operaciones para el tamaño concreto del enunciado, sin expresarlo en función de n:

In [ ]:
n_expresiones = math.perm(9, K) * math.factorial(4)   # 362.880
ops_por_expresion = K - 1                              # 4 operaciones aritméticas por expresión
total_operaciones = n_expresiones * ops_por_expresion

print(f"Para n=9 (caso del enunciado): {total_operaciones:,} operaciones aritméticas básicas en total")

**Justificación.** Esta forma de responder tiene la ventaja de ser extremadamente precisa para el caso concreto que plantea el
enunciado: en vez de una cota asintótica, da el número exacto de operaciones aritméticas que realmente se
ejecutan (1.451.520 para n=9), lo cual es información igual de correcta y más concreta que un simple "O(n⁵)".
Es apropiada si lo que interesa es el coste real de ejecutar el algoritmo tal como está planteado, sin
generalizarlo. Su límite frente a las opciones 1 y 2 es que, al no expresarse en función de un tamaño n que
pueda variar, no permite predecir cómo escalaría el algoritmo si el conjunto de cifras disponibles creciera
— que es justo la comparación que las Preguntas 5 y 7 necesitan para contrastar fuerza bruta con poda.

---
## Pregunta 6 — Algoritmo mejorado: backtracking con poda

`celdas 17–19` · Enunciado ✔ · Texto ✔ · Código ✔ · **Obligatoria (\*)**

### Validación

El diseño es correcto y bien argumentado: construye la expresión cifra a cifra y, cada vez que se cierra un
término, calcula una cota admisible (`max(suma, producto)` de las cifras más grandes que aún caben en los
huecos restantes) para decidir si la rama todavía puede batir el mejor máximo o mínimo conocidos. Documenta
además un diseño descartado con datos reales: una cota más floja que hacía que la poda fuera más lenta que
la fuerza bruta. Inconsistencia menor: la comparación de la cota usa división en coma flotante, algo
inofensivo aquí (solo afecta a una desigualdad de poda, no a la igualdad final exacta) pero que convendría
aclarar explícitamente dado el argumento "sin floats" de la Pregunta 2.

**Opción 1 — backtracking con cota admisible de dos lados, max y min combinados (así está)**

Construir la expresión incrementalmente, podando cuando ni el máximo ni el mínimo conocidos pueden mejorarse:

In [ ]:
def cota(cifras_restantes, huecos_que_faltan):
    if huecos_que_faltan <= 0:
        return 0
    elegidas = sorted(cifras_restantes, reverse=True)[:huecos_que_faltan]
    return max(sum(elegidas), math.prod(elegidas))

# En cada nodo del backtracking, tras cerrar un termino:
#   valor_cerrado = valor ya construido
#   c = cota(cifras aun libres, huecos que faltan por rellenar)
#   si valor_cerrado + c < mejor_max  y  valor_cerrado - c > mejor_min:
#       podar esta rama (ni el mejor ni el peor caso restante pueden mejorar nada)

**Justificación.** Es la opción más eficaz de las tres porque mantiene ambos objetivos (máximo y mínimo) en una única pasada
por el árbol de búsqueda, aprovechando que descartar una rama por no poder mejorar el máximo también sirve,
en el mismo paso, para evaluar si tampoco puede mejorar el mínimo. La cota es admisible porque, al ser todas
las cifras ≥ 1, ninguna combinación de +,−,×,÷ puede superar en valor absoluto a la suma o al producto de
esas mismas cifras. Reduce el espacio realmente explorado (no solo la constante) y está validada contra la
fuerza bruta para varios tamaños de n. Su límite es que sigue exigiendo, en el peor caso teórico, recorrer
todo el árbol si la cota nunca resulta lo bastante ajustada como para descartar ramas.

**Opción 2 — *meet-in-the-middle*: dividir la expresión en dos mitades**

Precalcular los valores posibles de cada mitad de la expresión por separado y combinarlos con tablas hash:

In [ ]:
def valores_mitad(cifras_disp, n_cifras_mitad, ops_disp):
    # Genera {valor: (cifras, operaciones)} para todas las sub-expresiones
    # posibles usando 'n_cifras_mitad' cifras y 'n_cifras_mitad - 1' operadores
    # de los disponibles. Se llama una vez para la mitad izquierda y otra
    # para la derecha con los operadores restantes, y luego se combinan
    # ambos diccionarios (con un operador +/- central) para obtener el total.
    resultados = {}
    for cifras in itertools.permutations(cifras_disp, n_cifras_mitad):
        for ops in itertools.permutations(ops_disp, n_cifras_mitad - 1):
            num, den = evaluar_exacto(cifras, ops)
            resultados.setdefault((num, den), (cifras, ops))
    return resultados

**Justificación.** Esta técnica precalcula todas las combinaciones posibles de "la primera mitad" y de "la segunda mitad" de la
expresión por separado, y las combina mediante una tabla hash en vez de explorar el árbol completo de una
vez. Es una alternativa igual de correcta y particularmente potente para la variante mencionada en la
Pregunta 11 ("¿existe una expresión que dé exactamente el valor V?"), donde permite responder en tiempo
proporcional a la raíz del espacio de búsqueda en vez de al espacio completo. Su límite frente a la opción 1
para este problema concreto es que añade complejidad de implementación (gestionar dos mitades, combinar
resultados, tratar el operador central que las une) que no aporta ninguna ventaja adicional cuando el
objetivo es solo encontrar el máximo y el mínimo con apenas 5 huecos.

**Opción 3 — dos pasadas de poda unilaterales, una para el máximo y otra para el mínimo**

Ejecutar el backtracking con poda dos veces, cada una optimizando un único extremo:

In [ ]:
def buscar_un_extremo_poda(pool, buscar_max, k=K):
    pool = list(pool); n = len(pool)
    mejor = {'valor': -math.inf if buscar_max else math.inf, 'expr': None}

    def mejora(v):
        return v > mejor['valor'] if buscar_max else v < mejor['valor']

    def cota_unilateral(restantes, huecos):
        return cota(restantes, huecos)   # misma cota admisible que en la opcion 1

    # ... misma logica recursiva que buscar_extremos_poda, pero la condicion
    # de poda solo compara contra mejor['valor'] (un unico extremo), no contra dos
    return mejor['valor'], mejor['expr']

maximo, expr_max = buscar_un_extremo_poda(CIFRAS, buscar_max=True)
minimo, expr_min = buscar_un_extremo_poda(CIFRAS, buscar_max=False)

**Justificación.** Esta variante es más sencilla de razonar que la opción 1 porque cada llamada persigue un único objetivo (solo
el máximo, o solo el mínimo), con una única cota y una única condición de poda por pasada, lo que la hace
más fácil de depurar paso a paso si algo falla. Sigue siendo un backtracking con poda correcto: encuentra
exactamente los mismos valores que la versión combinada. Su desventaja es de eficiencia: recorre el árbol de
búsqueda dos veces en lugar de una, y cada pasada individual dispone de menos información para podar
temprano, porque no se beneficia de que encontrar un buen candidato al buscar el máximo también ayude a
descartar ramas al buscar el mínimo — en la práctica, más lenta que la versión combinada de la opción 1.

---
## Pregunta 7 — Complejidad del algoritmo mejorado

`celdas 20–22` · Enunciado ✔ · Texto ✔ · Código + gráfica ✔ · **Obligatoria (\*)**

### Validación

Reconoce correctamente que una cota admisible no cambia la clase de peor caso — sigue siendo O(n⁵) —, porque
garantiza no perder el óptimo pero no garantiza recortar nada si el óptimo real está siempre cerca del
límite. A la vez, demuestra la mejora práctica con una tabla de speedups crecientes (3,0× en n=9 hasta ~5,5×
en n=15) generada y validada por código, no solo tecleada a mano, y conecta el resultado con el dato
negativo de la Pregunta 6 (la cota floja que hacía todo más lento).

**Opción 1 — peor caso O(n⁵) argumentado + mejora práctica medida y creciente (así está)**

Comparar tiempos de fuerza bruta y poda para varios n, validando que ambos dan el mismo resultado:

In [ ]:
n_valores = [9, 11, 13, 15]
t_bf, t_poda = [], []
for n in n_valores:
    pool = tuple(range(1, n + 1))
    t0 = time.perf_counter(); bf = fuerza_bruta_extremos(pool); t1 = time.perf_counter()
    t2 = time.perf_counter(); pv = buscar_extremos_poda(pool)[:2]; t3 = time.perf_counter()
    assert bf == pv
    t_bf.append(t1 - t0); t_poda.append(t3 - t2)
    print(f"n={n}: bf={t1-t0:.3f}s  poda={t3-t2:.3f}s  speedup={(t1-t0)/(t3-t2):.1f}x")

**Justificación.** Es la respuesta más honesta y completa: distingue con precisión "cota de peor caso" (que no mejora, sigue
siendo O(n⁵), porque una cota admisible nunca garantiza podar nada en el caso adversario) de "ganancia
empírica media" (que sí mejora, y de forma creciente con n, según se mide directamente). Validar en cada
medición que fuerza bruta y poda coinciden (`assert bf == pv`) añade una garantía de corrección que una
simple tabla de tiempos no da por sí sola. Es la única de las tres opciones que no exagera la mejora
conseguida, lo cual es especialmente importante en una pregunta obligatoria sobre un problema pequeño donde
las afirmaciones flojas se notan más.

**Opción 2 — intento de cota de peor caso más ajustada que O(n⁵)**

Modelar qué fracción del árbol se poda en función de n, para argumentar un límite superior más fino:

In [ ]:
# Idea: si en cada nodo la probabilidad de podar una rama fuera p(n),
# el numero de nodos visitados seria aproximadamente V(n,5)*4! * (1-p(n))^profundidad.
# Estimar p(n) empiricamente y proponer una cota superior mas ajustada que O(n^5),
# por ejemplo O(n^5 / g(n)) para alguna g(n) creciente observada en los datos.
def estimar_fraccion_podada(pool):
    total_nodos = 0     # se instrumentaria buscar_extremos_poda para contar nodos visitados
    total_posible = math.perm(len(pool), K) * math.factorial(4)
    # ... contar nodos realmente visitados con un contador global ...
    return total_nodos, total_posible

**Justificación.** Esta vía busca ir un paso más allá del peor caso teórico habitual, modelando cuántos nodos del árbol de
búsqueda se descartan realmente en función de n, con el objetivo de proponer una cota superior más fina que
O(n⁵). Es una dirección de análisis legítima y más ambiciosa. Su límite frente a la opción 1 es que una cota
de peor caso realmente más ajustada para poda con datos dependientes del problema concreto es difícil de
demostrar con rigor matemático completo; sin llevar la prueba hasta el final, la propuesta queda como una
estimación razonada pero no verificada, que debe presentarse explícitamente como tal y no como una cota
demostrada.

**Opción 3 — caracterización solo de la complejidad empírica/media, sin argumento de peor caso**

Ajustar una curva a los tiempos medidos del algoritmo con poda y quedarse solo con esa conclusión práctica:

In [ ]:
import numpy as np

ns = [9, 11, 13, 15]
tiempos_poda = [t_poda[i] for i in range(len(ns))]   # reutilizando la medicion de la opcion 1
pendiente_poda = np.polyfit(np.log(ns), np.log(tiempos_poda), 1)[0]
print(f"Exponente empírico observado con poda: {pendiente_poda:.2f}")

**Justificación.** Esta forma de responder documenta con claridad lo que realmente se observa en la práctica para los tamaños
probados, y tiene la ventaja de ser simple de calcular reutilizando las mismas mediciones de tiempo ya
generadas. Es una caracterización honesta y útil del comportamiento medio del algoritmo. Su límite frente a
la opción 1 es que, al no incluir el argumento de peor caso, deja sin responder qué pasaría en una entrada
adversaria donde la cota casi nunca permitiera podar (por ejemplo, cifras muy concentradas en valores
similares) — ese peor caso sigue siendo O(n⁵), y omitirlo deja la respuesta incompleta para una pregunta
marcada como obligatoria.

---
## Pregunta 8 — Juego de datos de entrada aleatorios

`celdas 23–25` · Enunciado ✔ · Texto ✔ · Código ✔ · Opcional

### Validación

El enfoque tiene sentido: generaliza el conjunto fijo {1..9} a n cifras positivas distintas cualesquiera,
variando tanto el tamaño n (para poner a prueba la predicción O(n⁵)) como el rango de valores. Usa
`random.sample` (sin reemplazo, cumple "sin repetir") y semillas fijas por reproducibilidad. Sugerencia
menor: añadir un caso límite deliberado, n=5 (el mínimo posible), daría más valor diagnóstico.

**Opción 1 — muestreo sin reemplazo, variando tamaño y rango, con semilla fija (así está)**

Generar un conjunto de n cifras distintas al azar, reproducible gracias a la semilla:

In [ ]:
import random

def generar_pool_aleatorio(n, rango_max=99, semilla=None):
    rnd = random.Random(semilla)
    return tuple(rnd.sample(range(1, rango_max + 1), n))

juegos_de_prueba = {
    "original (enunciado)":         CIFRAS,
    "n=12 aleatorio":               generar_pool_aleatorio(12, semilla=1),
    "n=16 aleatorio":               generar_pool_aleatorio(16, semilla=2),
    "n=14 aleatorio, rango amplio": generar_pool_aleatorio(14, rango_max=200, semilla=3),
}

**Justificación.** Es la opción más adecuada para este propósito: `random.sample` garantiza por construcción que no haya
cifras repetidas, cumpliendo la restricción del enunciado sin necesidad de comprobarlo aparte, y fijar la
semilla hace que cada ejecución del notebook reproduzca exactamente los mismos datos de prueba, lo cual es
esencial para poder comparar fuerza bruta y poda de forma fiable y para depurar si aparece alguna
discrepancia. Variar tanto el tamaño n como el rango de valores permite poner a prueba dos hipótesis
distintas a la vez: la predicción de complejidad O(n⁵) y la independencia del rendimiento respecto a la
magnitud de las cifras. Su único límite es que, al ser aleatorio, no garantiza cubrir por sí solo los casos
límite del dominio (por ejemplo, el tamaño mínimo n=5).

**Opción 2 — suite de casos deliberados (mínimo, original, estrés)**

Elegir explícitamente los tamaños de prueba en vez de sortearlos, cubriendo los extremos del dominio:

In [ ]:
def generar_pool_aleatorio(n, rango_max=99, semilla=None):
    rnd = random.Random(semilla)
    return tuple(rnd.sample(range(1, rango_max + 1), n))

juegos_de_prueba = {
    "mínimo posible (n=5)": generar_pool_aleatorio(5, semilla=10),   # sin ninguna libertad de eleccion
    "original (n=9)":       CIFRAS,
    "estrés (n=20)":        generar_pool_aleatorio(20, semilla=11),
}

**Justificación.** Esta variante combina lo mejor de ambos mundos: sigue usando `random.sample` (por lo que las cifras dentro
de cada conjunto son aleatorias y sin repetición), pero elige a propósito los tamaños n a probar en vez de
sortearlos también, asegurando que se cubran los bordes del dominio válido — el caso mínimo n=5, donde no
hay ninguna libertad de elección de cifras, y un caso de estrés n=20 para observar el crecimiento
predicho. Da más valor diagnóstico por caso que el muestreo puramente aleatorio de tamaños. Su límite es
que, al fijar los tamaños de antemano, es menos "aleatorio" en el sentido literal que pide la pregunta —
funciona mejor combinada con la opción 1 que como sustituto único.

**Opción 3 — mismo muestreo sin reemplazo, sin fijar semilla**

Generar las cifras al azar de la misma forma correcta, pero sin fijar semilla para reproducibilidad:

In [ ]:
def generar_pool_aleatorio_sin_semilla(n, rango_max=99):
    return tuple(random.sample(range(1, rango_max + 1), n))   # sigue sin repetir cifras

juego_de_prueba = generar_pool_aleatorio_sin_semilla(12)

**Justificación.** Esta versión respeta igual de bien la restricción "sin repetir" que la opción 1 — sigue usando
`random.sample`, no `random.randint` —, así que el conjunto generado es tan válido matemáticamente como el
de las otras dos opciones, y es ligeramente más simple porque no hay que gestionar ni documentar ninguna
semilla. Tiene sentido para pruebas rápidas y exploratorias donde no importa poder repetir exactamente el
mismo experimento. Su límite frente a la opción 1 es precisamente ese: al no fijar semilla, cada ejecución
del notebook genera un juego de datos distinto, lo que dificulta reproducir exactamente los mismos
resultados al comparar fuerza bruta y poda, o al depurar si apareciera alguna discrepancia entre ambos
algoritmos.

---
## Pregunta 9 — Aplicación del algoritmo al juego de datos

`celdas 26–28` · Enunciado ✔ · Texto ✔ · Código ✔ · Sin gráfica · Opcional

### Validación

Buena práctica de verificación: aplica ambos algoritmos (fuerza bruta y poda) a cada juego de datos y
comprueba que coinciden en mínimo y máximo, reportando además cuántos enteros del rango quedan sin cubrir
("huecos"). Es la única sección de resultados sin ninguna ilustración, mientras que las Preguntas 5 y 7 sí
incluyen gráficas — el concepto de "huecos" es visual por naturaleza y se explicaría mejor con un gráfico.

**Opción 1 — doble ejecución + validación cruzada + conteo de huecos (así está)**

Correr fuerza bruta y poda sobre cada juego de datos y comparar resultados:

In [ ]:
for nombre, pool in juegos_de_prueba.items():
    t0 = time.perf_counter(); res = fuerza_bruta(pool); t_bf = time.perf_counter() - t0
    mn, mx = min(res), max(res)
    t0 = time.perf_counter(); mn2, mx2, _, _ = buscar_extremos_poda(pool); t_poda = time.perf_counter() - t0
    ok = "OK" if (mn, mx) == (mn2, mx2) else "DIFERENCIA!"
    huecos = [v for v in range(mn, mx + 1) if v not in res]
    print(f"{nombre:32s} min={mn} max={mx} huecos={len(huecos)} "
          f"t_bruta={t_bf:.3f}s t_poda={t_poda:.3f}s {ok}")

**Justificación.** Es la opción más convincente para una entrega académica porque no se conforma con la validación ya hecha una
vez en la Pregunta 7: repite la comprobación cruzada entre fuerza bruta y poda en cada dataset nuevo,
detectando de inmediato si alguna diferencia apareciera en tamaños o rangos no probados antes. Reportar
también el número de huecos por caso conecta directamente con la discusión de cobertura de la Pregunta 3.
Su único límite es que toda la información se presenta como texto/tabla impresa, sin ningún apoyo visual,
pese a que el concepto de "huecos" es inherentemente espacial y se explicaría con más claridad en un
gráfico.

**Opción 2 — lo mismo, más un gráfico de recta numérica con los huecos marcados**

Añadir una visualización explícita de qué enteros se alcanzan y cuáles no, para el caso más pequeño:

In [ ]:
import matplotlib.pyplot as plt

nombre, pool = "original (enunciado)", CIFRAS
res = fuerza_bruta(pool)
mn, mx = min(res), max(res)
alcanzados = [v for v in range(mn, mx + 1) if v in res]
huecos = [v for v in range(mn, mx + 1) if v not in res]

plt.eventplot(alcanzados, colors='tab:blue', lineoffsets=1)
plt.eventplot(huecos, colors='tab:red', lineoffsets=1)
plt.yticks([]); plt.xlabel('valor entero')
plt.title('Enteros alcanzables (azul) vs. huecos (rojo)')
plt.show()

**Justificación.** Esta opción parte exactamente de la misma validación cruzada de la opción 1 y le añade una recta numérica
que marca en azul los enteros alcanzables y en rojo los huecos, haciendo visible de un vistazo algo que
antes solo aparecía como un número (`len(huecos)`). Es la forma más completa de responder porque cierra el
único punto pendiente del checklist de ilustraciones para esta pregunta, sin sacrificar nada de la validación
ya presente. Su único costo es una gráfica adicional por generar y explicar, y que su valor es mayor para
los juegos de datos más pequeños (para n grandes, el rango de huecos puede ser demasiado ancho para
visualizarse con claridad en una sola recta).

**Opción 3 — aplicar solo el algoritmo con poda, confiando en la validación previa**

Ejecutar únicamente la versión rápida sobre los nuevos juegos de datos, sin repetir la fuerza bruta:

In [ ]:
for nombre, pool in juegos_de_prueba.items():
    t0 = time.perf_counter()
    mn, mx, expr_min, expr_max = buscar_extremos_poda(pool)
    t_poda = time.perf_counter() - t0
    print(f"{nombre:32s} min={mn} max={mx}  t_poda={t_poda:.3f}s")

**Justificación.** Esta forma de aplicar el algoritmo tiene la ventaja de ser más rápida de ejecutar sobre datasets grandes,
ya que evita repetir la fuerza bruta completa (que crece como O(n⁵)) en cada uno de ellos, apoyándose en que
la Pregunta 7 ya demostró que la poda coincide con la fuerza bruta para varios tamaños. Es razonable cuando
el tiempo de cómputo es una restricción real. Su límite frente a las opciones 1 y 2 es que pierde la
comprobación cruzada en cada dataset nuevo — precisamente el tipo de verificación que en la Pregunta 6
detectó que una primera versión de la cota era defectuosa —, así que para una entrega académica donde lo que
se valora es demostrar corrección, no solo velocidad, es la opción menos convincente de las tres.

---
## Pregunta 10 — Referencias utilizadas

`celdas 29–30` · Enunciado ✔ · Texto ✔ · Código no aplica · Opcional

### Validación

Lista razonable (enunciado del trabajo, documentación de `itertools`/`math`, CLRS, el "juego del 24"/Krypto
como inspiración de la cota). No hay nada técnicamente incorrecto que validar aquí — es una bibliografía, no
una afirmación verificable. El único criterio real es la honestidad: el propio rubric dice "si ha sido
necesario", así que solo debería listarse lo que de verdad se consultó. Aquí no aplica código, así que las
tres alternativas son de redacción, no de implementación.

**Opción 1 — lista específica y verificable (así está).** Enunciado del trabajo, documentación oficial de
Python, un libro de texto concreto (CLRS), y una técnica de inspiración nombrada explícitamente (el "juego
del 24"). **Justificación completa:** es la opción más útil para un corrector porque cada fuente es concreta
y rastreable — se puede verificar que `itertools` y `math` son en efecto las librerías usadas en el código, y
que la idea de "poda por cota admisible" tiene un paralelismo real con las estrategias conocidas para el
juego del 24. Tiene sentido mantenerla siempre que esas fuentes se hayan consultado de verdad; citar una
referencia no utilizada, aunque sea plausible, sería menos honesto que no citarla.

**Opción 2 — declarar explícitamente que no hizo falta nada más allá del enunciado.** *"No ha sido necesario
consultar ninguna referencia externa más allá del enunciado del ejercicio y la documentación estándar de
Python (`itertools`, `math`)."* **Justificación completa:** el propio rubric admite "si ha sido necesario",
así que una frase honesta y mínima es una respuesta perfectamente completa si en la práctica no se abrió
ningún libro de texto ni se buscó ninguna técnica externa — sencillamente porque el problema, con la
interpretación K=5 adoptada, no lo requería. Preferir esta opción sobre inflar la lista con referencias
generales no consultadas es la práctica académicamente más honesta, y no resta ningún punto frente a una
lista más larga si es la verdad.

**Opción 3 — bibliografía basada en el material de curso, sin buscar fuentes externas.** *"Apuntes y
diapositivas del Seminario de Algoritmos de Optimización (VIU 03MIAR); documentación de la librería estándar
de Python consultada durante el desarrollo (`itertools.permutations`, `math.perm`, `math.factorial`,
`math.prod`)."* **Justificación completa:** esta opción sitúa la referencia principal en el propio material
de la asignatura en vez de en fuentes externas de investigación, lo cual es realista para un trabajo de
seminario que no pretende aportar bibliografía académica nueva sino aplicar lo enseñado en clase. Es tan
válida como las otras dos y evita el riesgo de citar un libro de texto (como CLRS) que en realidad no se
llegó a abrir, limitándose a las fuentes efectivamente consultadas durante la implementación.

---
## Pregunta 11 — Líneas futuras de estudio

`celdas 31–32` · Enunciado ✔ · Texto ✔ · Código no aplica · Opcional

### Validación

Respuesta completa y bien calibrada al rubric, que pide explícitamente cubrir tanto variantes del problema
como variantes de tamaño. Cubre ambas: paréntesis, repetición/otros operadores, el problema inverso vía
*meet-in-the-middle*, programación dinámica sobre subconjuntos, paralelización, y escalado de tamaño. Cada
punto trae su propio "por qué", no es solo una lista de palabras clave. Tampoco aplica código aquí: las tres
alternativas son de enfoque y alcance, no de implementación.

**Opción 1 — panorama amplio: seis líneas concretas, cada una con su justificación técnica (así está).**
Paréntesis (cambiaría el espacio de soluciones a algo del orden de los números de Catalan), repetición u
operadores adicionales, el problema inverso ("¿existe una expresión que dé el valor V?") resuelto vía
*meet-in-the-middle*, programación dinámica sobre subconjuntos de cifras, paralelización con
`multiprocessing`, y escalado del tamaño n. **Justificación completa:** responde literalmente a las dos
partes que pide el rubric —variaciones del problema y variaciones de tamaño— con profundidad técnica real en
cada punto, no genérica; cada línea conecta con algo concreto ya construido en el notebook (la cota de la
Pregunta 6, la complejidad de la Pregunta 7), lo que demuestra que las líneas futuras nacen de haber
entendido a fondo la solución actual, no de una lista de ideas sueltas.

**Opción 2 — foco profundo en una sola línea, desarrollada con un prototipo.** Elegir únicamente la variante
del problema inverso ("¿existe una expresión que dé exactamente el valor V?") y esbozar directamente el
diseño de un *meet-in-the-middle* para resolverla, incluyendo su complejidad esperada (aproximadamente
O(√(espacio total)) frente al O(n⁵) de recorrer todo el espacio). **Justificación completa:** demuestra más
profundidad técnica en una sola idea que la opción 1, incluyendo un análisis de complejidad propio para la
variante propuesta, lo cual puede ser más convincente si lo que se valora es la capacidad de desarrollar una
idea hasta el final en vez de enumerar posibilidades. Es tan válida como la opción 1; su única diferencia es
de alcance: cubre menos líneas distintas porque invierte ese espacio en profundizar una sola.

**Opción 3 — priorizar variantes de tamaño y escalabilidad de ingeniería sobre variantes matemáticas del
problema.** Centrar la respuesta en cómo escalar la solución actual a instancias más grandes: paralelización
con `multiprocessing` repartiendo las variaciones de cifras entre procesos, uso de `numpy`/vectorización para
evaluar lotes de expresiones a la vez, y perfilado para identificar el verdadero cuello de botella antes de
optimizar. **Justificación completa:** es una respuesta igual de legítima que las otras dos, con un enfoque
más práctico y de ingeniería de rendimiento que matemático, útil si el interés real está en llevar esta
solución concreta a instancias de tamaño mucho mayor sin cambiar la naturaleza del problema. Su diferencia
frente a la opción 1 es de énfasis: no explora variantes matemáticas del enunciado (paréntesis, repetición,
otros operadores), así que responde con más fuerza a la mitad de "variaciones al alza del tamaño" que a la
de "variaciones del problema" que también pide el rubric.

---
## Resumen final

| # | Pregunta | Tipo | Enunciado | Texto | Código | Veredicto |
|---|----------|------|:---:|:---:|:---:|-----------|
| 0 | Descripción del problema | — | ❌ | — | — | Falta pegar el enunciado real (placeholder sin completar) |
| 1 | Posibilidades sin/con restricciones | (*) + opc. | ✅ | ✅ | ✅ | Correcta; añadir cita explícita del ejemplo como justificación de K=5 |
| 2 | Estructura de datos | (*) | ✅ | ✅ | ✅ | Correcta y bien justificada, incl. diseño descartado |
| 3 | Función objetivo / max·min | (*)×2 | ✅ | ✅ | ✅ | Sólida; añadir la faceta de "cobertura" como tercer matiz |
| 4 | Diseño fuerza bruta | Opc. | ✅ | ✅ | ✅ | Correcta; aclarar que las optimizaciones no cambian la clase O() |
| 5 | Complejidad fuerza bruta | Opc. | ✅ | ✅ | ✅ | Correcta, rigurosa, validada con gráfica |
| 6 | Algoritmo mejorado (poda) | (*) | ✅ | ✅ | ⚠️ | Muy bien justificada; poda usa floats de forma inconsistente con Q2 (inofensivo, aclarar) |
| 7 | Complejidad del mejorado | (*) | ✅ | ✅ | ✅ | Correcta y honesta (no exagera la mejora asintótica) |
| 8 | Datos aleatorios | Opc. | ✅ | ✅ | ✅ | Correcta; podría añadir un caso límite n=5 |
| 9 | Aplicación a los datos | Opc. | ✅ | ✅ | ✅ | Correcta; única sección sin gráfica — añadir una |
| 10 | Referencias | Opc. | ✅ | ✅ | — | Adecuada si es honesta sobre lo realmente consultado |
| 11 | Líneas futuras | Opc. | ✅ | ✅ | — | Completa, cubre variantes y escalado como pide el rubric |

### Sobre lenguaje, comentarios e ilustraciones (punto 5 del checklist)

- **Lenguaje:** claro y técnico en todo el notebook, sin ambigüedad. Bien.
- **Código comentado:** las funciones clave (`evaluar_exacto`, `cota`, `buscar_extremos_poda`,
  `fuerza_bruta`) tienen docstrings que explican el *porqué*, no solo el qué. Bien.
- **Ilustraciones:** dos gráficas presentes (fuerza bruta vs. n⁵ en la Pregunta 5; fuerza bruta vs. poda en
  la Pregunta 7). Falta una en la Pregunta 9 — es el único punto pendiente en este apartado.

---
*Validación basada en lectura estática del notebook (código y texto), no en una ejecución real de las
celdas. Antes de entregar, ejecuta todo `Resuelto.ipynb` de arriba a abajo una vez y confirma que las
cifras impresas (tiempos, speedups, mínimo/máximo) coinciden con las que aparecen escritas en las celdas de
texto — especialmente la tabla de la Pregunta 7, que está tecleada a mano y debe coincidir con lo que
produce la celda de código inmediatamente después.*